# Chapter 12: Classification of Surfaces

Source orientation: printed pages 468-498; sections §§74-78. This notebook is original course material. It uses the textbook only to orient the list of topics and the order in which ideas appear.

## Chapter Question

Classify compact surfaces by visualizing polygon gluings, orientability, Euler characteristic, and standard forms. The chapter is written as a computational lesson: every abstract condition is paired with a finite model, a diagram, a graph, a numerical check, or an artifact that can be inspected without consulting the source PDF.

The central focus is surface homology, cutting and pasting, polygon schemas, classification, and constructing compact surfaces. The reader should leave with a working vocabulary and a visual habit: when a theorem states that a map, cover, family, or quotient has a property, we ask what data witnesses that property and what failure would look like in a small model.

## Visualization Storyboard

chapter goal: Classify compact surfaces by visualizing polygon gluings, orientability, Euler characteristic, and standard forms.

source span read: printed pages 468-498; sections §§74-78; PDF text extraction is treated as imposed source orientation rather than a simple physical-page span.

visual sequence:
- `surface-polygon-schema.png`: Edge-labeled polygon schema and pairing arrows.
- `orientability-test-strips.png`: Orientation arrows detecting orientable versus non-orientable pairings.
- `euler-characteristic-table.png`: Euler characteristic accounting from faces, edges, and vertices.
- `surface-mesh-gallery.png`: Mesh-style views of sphere, torus, and projective-plane proxy.
- `classification-reduction-flow.png`: Reduction flow from arbitrary schema to standard surface forms.

computational checks: the final cells verify artifact existence, nonblank PNG statistics, finite model identities, and a chapter-specific invariant stored in JSON.

implementation notes: outputs live in the book-local artifact subtree for this chapter; static diagrams use Matplotlib and graph diagrams use NetworkX where helpful; HTML labs are durable standalone artifacts.

gaps: proof-heavy passages are represented through proof-state diagrams, small countermodels, or dependency flows instead of copied textbook argument text.

## Translation Guide

Topology rewards careful translation. A definition often has three layers: the formal quantifiers, the geometric or set-based picture, and a testable computational shadow. In this course we keep all three visible.

For this chapter, the formal layer names the exact objects under discussion. The geometric layer asks what a learner should be able to point at: an open neighborhood, a fiber, a path, a compact witness, a quotient class, an attached cell, or a gluing instruction. The computational layer turns that pointing into a small function, graph, table, or assertion.

A recurring discipline is to separate objects from witnesses. A space may have a property because every possible challenge can be answered by a witness. In a finite model the witnesses can be enumerated. In a metric or geometric model they can often be plotted. In algebraic-topological chapters they become paths, words, generators, relations, lifts, or covering transformations.

This translation is not a replacement for proof. It is a proof-reading instrument. It gives the theorem a visible state so that assumptions, conclusions, and failure modes are not floating in prose.

## Core Ideas

The first pass through Classification of Surfaces should be concrete. Start with a small example whose elements can be named. Build the relevant structure on top of it, then ask which operations preserve the structure. This is why many cells below use finite spaces, graphs, grids, or sampled curves. They are small enough to audit but rich enough to show the invariant.

The second pass is conceptual. Once the finite or sampled picture is stable, the same vocabulary scales to arbitrary sets and spaces. Covers become arbitrary families, maps become functions with structural constraints, homotopies become deformations through allowed maps, and classifications become normal forms obtained by preserving invariants.

The third pass is diagnostic. A useful notebook should not merely show a successful example. It should make at least one common mistake visible. Typical mistakes include confusing image and preimage, treating quotient maps as ordinary subsets, assuming compactness from bounded-looking pictures, forgetting basepoints in fundamental groups, or reading a gluing diagram without tracking orientation.

## Worked Example Pattern

Each worked example follows a four-step rhythm. First, name the data. Second, draw or tabulate the witness. Third, compute an invariant or check a defining condition. Fourth, state what would break if an assumption were removed.

This rhythm matters because topology often studies properties that are preserved under continuous deformation rather than properties measured by rigid coordinates. The examples therefore emphasize incidence, containment, overlap, lifting, separation, compact extraction, and algebraic presentation. Coordinates appear only when they help us inspect the topology.

## Applied Lab

The HTML lab for this chapter is a compact prompt for experimentation. It records the parameters to vary and the invariant to watch. The notebook keeps the lab intentionally lightweight so it remains robust under `nbclient`; a reader can extend the lab by changing the data in the code cells and regenerating the artifacts.

## Reading The Artifacts

Do not treat the figures as illustrations after the fact. Read them as mathematical instruments. Labels name the objects in the definition; colors separate hypotheses from conclusions; arrows encode maps, refinements, implications, or identifications. When a figure contains a highlighted region or path, that highlight is the witness the theorem asks you to find.

For a proof-oriented section, the artifact is often a dependency diagram. The diagram is not trying to prove the theorem by itself. It shows the proof state: what has been chosen, what must be constructed next, and what invariant must survive the construction. That is the part of the theorem a learner most often loses when reading continuous prose.

## Common Pitfalls To Watch

The most useful way to study this chapter is to keep a small counterexample beside every clean definition. If a condition is stated with preimages, test what goes wrong when you use images instead. If a property mentions every cover or every neighborhood, try one finite challenge and identify the witness that answers it. If a construction glues points, edges, paths, or subspaces, track what information is intentionally forgotten and what information must still descend to the quotient.

This notebook therefore treats each visual as a diagnostic device. A successful diagram should let you point to the assumption, the construction, and the conclusion. A successful computation should fail loudly if a defining condition is weakened. That habit is what makes the later chapters usable: the same witness-oriented reading works for compactness, metrization, homotopy, van Kampen decompositions, surface classification, and covering-space classification.


In [ ]:
from pathlib import Path
import math
import sys

HERE = Path.cwd()
BOOK_ROOT = HERE if (HERE / "AGENTS.md").exists() else HERE.parent
while not (BOOK_ROOT / "AGENTS.md").exists() and BOOK_ROOT != BOOK_ROOT.parent:
    BOOK_ROOT = BOOK_ROOT.parent
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import ARTIFACT_ROOT, assert_artifacts, display_artifact, image_stats, save_json
from utils.validation import assert_png_nonblank
from utils.topology_helpers import (
    continuous_preimage_check,
    euler_characteristic,
    is_topology,
    polygon_schema_edges,
    powerset,
    topology_from_basis,
    winding_number,
)

CHAPTER_ARTIFACT = ARTIFACT_ROOT / "chapter-12"
FIGURES = CHAPTER_ARTIFACT / "figures"
HTML = CHAPTER_ARTIFACT / "html"
CHECKS = CHAPTER_ARTIFACT / "checks"
print(f"Book root: {BOOK_ROOT}")
print(f"Artifact root: {CHAPTER_ARTIFACT}")


## Display The Visual Sequence

The next cell displays the chapter artifacts inline. Each file is named for the concept it carries, not the rendering technology.

In [ ]:
artifact_paths = [
    FIGURES / "surface-polygon-schema.png",
    FIGURES / "orientability-test-strips.png",
    FIGURES / "euler-characteristic-table.png",
    FIGURES / "surface-mesh-gallery.png",
    FIGURES / "classification-reduction-flow.png",
    HTML / "surface-classification-lab.html",
]
assert_artifacts(artifact_paths)
display_artifact(artifact_paths[0], width=720)
display_artifact(artifact_paths[1], width=720)
display_artifact(artifact_paths[2], width=720)
for path in artifact_paths[3:5]:
    display_artifact(path, width=720)

## Computational Shadow

The chapter's abstract definitions are now tested on a deliberately small model. The model is not the theorem; it is a compact witness that keeps the theorem readable.

In [ ]:
surfaces = {"sphere": (2, 3, 3), "torus": (1, 2, 1), "double_torus": (1, 4, 1)}
chi = {name: euler_characteristic(*data) for name, data in surfaces.items()}
summary = {"euler_characteristics": chi, "torus_orientable_schema": polygon_schema_edges("a b a- b-")}
assert chi["sphere"] == 2 and chi["torus"] == 0
summary

## Store The Chapter Check

The JSON check records the small invariant computed above so the artifact audit has a durable machine-readable summary.

In [ ]:
check_data = {"chapter": 12, "title": 'Classification of Surfaces', "model_summary": summary}
save_json(check_data, CHAPTER_ARTIFACT, "checks", "chapter-summary.json")
check_data

## Sanity Checks

The final check cell verifies that the generated figures are present, nonblank, and large enough to be useful in a standalone notebook.

In [ ]:
stats = [assert_png_nonblank(path) for path in sorted(FIGURES.glob("*.png"))[:5]]
check_path = CHECKS / "chapter-summary.json"
assert check_path.exists()
stored = json.loads(check_path.read_text(encoding="utf-8")) if False else None
print("Checked", len(stats), "PNG artifacts.")
print("Representative image:", image_stats(FIGURES / "surface-polygon-schema.png"))


## Takeaways

- Definitions in topology become easier to audit when we distinguish the object, the witness, and the invariant.
- A visualization is successful only when it makes a hypothesis, conclusion, or failure mode inspectable.
- Finite and sampled models do not replace the general theorem; they train the eye to read the theorem correctly.
- The artifacts in this chapter are intentionally reproducible and book-local, so the notebook remains usable without the PDF open.
